# Bank Term Deposit Prediction — Classification Pipeline

This notebook builds and evaluates multiple **Classification** models to predict whether a bank client subscribes to a term deposit (**`deposit`**: `yes` = `1`, `no` = `0`) using **`bank.csv`**, structured in the exact style of the `stroke` classification pipeline.


## 1. Importing Libraries

Load data manipulation, visualization, preprocessing, classification models, and evaluation metric libraries.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier
)
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


## 2. Exploratory Data Analysis (EDA)

Load `bank.csv` and inspect dataset shape, preview rows, column types, missing values, duplicates, and target class distribution (`deposit`).


In [ ]:
df = pd.read_csv('bank.csv')
print('Shape:', df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nTarget class counts (deposit):')
print(df['deposit'].value_counts(dropna=False))
df.describe(include='all')


## 3. Data Visualization

Visualize numerical feature distributions, target class counts (`deposit`), and relationships with `balance`, `duration`, and `job`.


In [ ]:
num_plot_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, col in zip(axes.flat, num_plot_cols):
    sns.histplot(df[col].dropna(), bins=25, kde=True, ax=ax)
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
axes.flat[-1].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dep_counts = df['deposit'].dropna().value_counts()
axes[0].bar(dep_counts.index, dep_counts.values)
axes[0].set_title('Term Deposit Subscription (deposit) Counts')
axes[0].set_xlabel('deposit')
axes[0].set_ylabel('Count')
sns.boxplot(data=df.dropna(subset=['deposit']), x='deposit', y='duration', ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_title('Call Duration by Deposit Subscription (Log Scale)')
axes[1].set_xlabel('deposit')
axes[1].set_ylabel('duration (s)')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
top_jobs = df['job'].value_counts().head(6).index
sns.countplot(data=df[df['job'].isin(top_jobs)].dropna(subset=['deposit']), y='job', hue='deposit', ax=axes[0])
axes[0].set_title('Top Jobs by Deposit Subscription')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('job')
sns.countplot(data=df.dropna(subset=['deposit']), x='poutcome', hue='deposit', ax=axes[1])
axes[1].set_title('Previous Campaign Outcome by Deposit Subscription')
axes[1].set_xlabel('poutcome')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()


## 4. Data Preprocessing and Train/Test Split

Drop rows with missing `deposit` target (`9` rows), encode `deposit` (`yes` -> `1`, `no` -> `0`), perform a stratified 80/20 split (`random_state=42`), impute missing numerical and categorical features strictly on `X_train`, encode categories with `OrdinalEncoder`, and scale with `MinMaxScaler(feature_range=(0, 1))`.


In [ ]:
data = df.dropna(subset=['deposit']).copy()
X = data.drop('deposit', axis=1)
y = (data['deposit'].astype(str).str.strip().str.lower() == 'yes').astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print('X_train:', X_train.shape, 'X_test:', X_test.shape)
print('Positive class rate in y_train:', round(y_train.mean(), 4), 'y_test:', round(y_test.mean(), 4))


In [ ]:
num_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
cat_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')
X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])
X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols])
X_test[cat_cols] = encoder.transform(X_test[cat_cols])
scaler = MinMaxScaler(feature_range=(0, 1))
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


## 5. Model Training and Evaluation

Train nine classification models (`Logistic Regression`, `KNN`, `SVM`, `Gaussian NB`, `Decision Tree`, `Random Forest`, `Gradient Boosting`, `AdaBoost`, and `Extra Trees`) and compute `Accuracy`, `Precision`, `Recall`, `F1_Score`, `ROC_AUC`, and `PR_AUC` on the test set.


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
    'Gaussian NB': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
    'AdaBoost': AdaBoostClassifier(random_state=RANDOM_STATE),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=RANDOM_STATE),
}
results = []
preds_dict = {}
probs_dict = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = model.decision_function(X_test)
    preds_dict[name] = y_pred
    probs_dict[name] = y_prob
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1_Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_test, y_prob),
        'PR_AUC': average_precision_score(y_test, y_prob),
    })
results_df = pd.DataFrame(results).sort_values(by='ROC_AUC', ascending=False).reset_index(drop=True)
results_df


## 6. ROC Curves and Precision-Recall AUC Comparison

Plot ROC curves across all evaluated classifiers alongside a horizontal bar comparison of `ROC_AUC` and `PR_AUC`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for name in results_df['Model']:
    fpr, tpr, _ = roc_curve(y_test, probs_dict[name])
    auc_val = roc_auc_score(y_test, probs_dict[name])
    axes[0].plot(fpr, tpr, label=f'{name} (AUC = {auc_val:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', label='Random Baseline (AUC = 0.500)')
axes[0].set_title('ROC Curves Across All Models')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right', fontsize=8)
auc_df = results_df.set_index('Model')[['ROC_AUC', 'PR_AUC']]
auc_df.plot(kind='barh', ax=axes[1])
axes[1].invert_yaxis()
axes[1].set_title('ROC-AUC vs. PR-AUC Comparison')
axes[1].set_xlabel('Score')
axes[1].set_xlim(0, 1.0)
plt.tight_layout()
plt.show()


## 7. Confusion Matrices for All Classification Models

Display the confusion matrix heatmap for every trained classifier so true positives, true negatives, false positives, and false negatives can be inspected across all models.


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 13))
for ax, name in zip(axes.flat, models.keys()):
    cm = confusion_matrix(y_test, preds_dict[name])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    ax.set_title(f'{name}')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
plt.suptitle('Confusion Matrices Across All Classification Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 8. Classification Reports for All Classification Models

Print precision, recall, F1-score, and support for each class across every evaluated classifier.


In [ ]:
for name in models.keys():
    print(f'=== {name} ===')
    print(classification_report(y_test, preds_dict[name], digits=4, zero_division=0))


## 9. Final Conclusion

- **Dataset & Class Balance:** `bank.csv` contains `8,912` records and `17` columns (`0` duplicate rows), with missing values across multiple columns (highest in `marital` at `169`, `pdays` at `145`, and `deposit` at `9`), and balanced target classes (`4,673` `no` vs. `4,230` `yes` among non-null `deposit` entries).
- **Exploratory & Visualization Findings:** Call `duration`, account `balance`, and prior campaign outcome (`poutcome == 'success'`) show clear patterns associated with term deposit subscription, while `balance`, `duration`, and `campaign` exhibit right-skewed distributions.
- **Model Comparison:** **Gradient Boosting** achieved the highest `ROC_AUC` (`0.9082`), `PR_AUC` (`0.8965`), `Accuracy` (`0.8360`), and `F1_Score` (`0.8315`), followed by **Random Forest** (`ROC_AUC: 0.9016`, `Accuracy: 0.8310`) and **SVM** (`ROC_AUC: 0.8886`, `Accuracy: 0.8186`).
